# Gemma 4 Trust & Safety Context Pruning Runner

Use this notebook inside Kaggle for Gemma 4 Trust & Safety research experiments.
It runs the ACPA pipeline, creates benchmark artifacts, and writes JSONL output.

Expected Kaggle setup:

1. Attach the Agentic Eval dataset to this notebook.
2. Add your Gemma/Gemini API key through Kaggle Secrets or paste it into the config cell.
3. Run cells from top to bottom.


In [ ]:
from pathlib import Path
import json
import os
import sys

# If this notebook is in the repo, REPO_ROOT is the parent of notebooks/.
# In Kaggle, set REPO_ROOT to wherever you cloned or uploaded this repository.
possible_roots = [
    Path.cwd(),
    Path('/kaggle/working/context-pruning'),
    Path('/kaggle/input/context-pruning'),
]
REPO_ROOT = next((path for path in possible_roots if (path / 'src/acpa_gemma').exists()), Path.cwd())
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print('REPO_ROOT =', REPO_ROOT)
print('SRC_ROOT exists =', SRC_ROOT.exists())
print('Kaggle input exists =', Path('/kaggle/input').exists())


## Optional dependency install

Run this if Kaggle does not already have the required packages. Internet must be enabled in the notebook settings.


In [ ]:
# Uncomment when needed:
# !pip install -q google-genai pandas pyarrow tomli


## Locate Agentic Eval dataset

If auto-detection picks the wrong path, set `DATASET_DIR` manually to the attached dataset directory.


In [ ]:
input_root = Path('/kaggle/input')
if input_root.exists():
    candidates = sorted([p for p in input_root.iterdir() if p.is_dir()])
    print('Available /kaggle/input datasets:')
    for candidate in candidates:
        print(' -', candidate)
    DATASET_DIR = next((p for p in candidates if 'agent' in p.name.lower() and 'eval' in p.name.lower()), candidates[0] if candidates else input_root)
else:
    DATASET_DIR = Path('demo-missing-kaggle-input')

print('DATASET_DIR =', DATASET_DIR)


## Create config files

Preferred: add a Kaggle Secret named `GEMINI_API_KEY` or `GEMMA_API_KEY`.
The cell writes `/kaggle/working/configs/secrets.toml`, which is what the pipeline reads.


In [ ]:
config_dir = Path('/kaggle/working/configs') if Path('/kaggle/working').exists() else REPO_ROOT / 'configs'
config_dir.mkdir(parents=True, exist_ok=True)

api_key = ''
try:
    from kaggle_secrets import UserSecretsClient
    secrets_client = UserSecretsClient()
    for secret_name in ['GEMINI_API_KEY', 'GEMMA_API_KEY', 'GOOGLE_API_KEY']:
        try:
            api_key = secrets_client.get_secret(secret_name)
            if api_key:
                print(f'Loaded API key from Kaggle Secret: {secret_name}')
                break
        except Exception:
            pass
except Exception as exc:
    print('Kaggle Secrets unavailable:', exc)

# If you are not using Kaggle Secrets, paste your key below before the real Gemma run.
# api_key = 'YOUR_GOOGLE_AI_STUDIO_OR_GEMINI_API_KEY'

(config_dir / 'app.toml').write_text(f'''
[gemma]
model = "gemma-4-26b-a4b-it"
temperature = 0.2
top_p = 0.9
max_output_tokens = 2048

[data]
input_dir = "{DATASET_DIR}"
sample_size = 0

[pruning]
alpha = 1.5
beta = 1.0
gamma = 0.5
delta = 10.0
prune_ratio = 0.45
cache_threshold = 2
priority_boost = 1.5

[output]
path = "/kaggle/working/results.jsonl"
'''.strip() + '
', encoding='utf-8')

(config_dir / 'secrets.toml').write_text(f'''
[gemma]
api_key = "{api_key}"
'''.strip() + '
', encoding='utf-8')

print('Wrote', config_dir / 'app.toml')
print('Wrote', config_dir / 'secrets.toml')
print('API key present =', bool(api_key))


## Dry-run pipeline test, no API key required

This verifies dataset loading, context construction, ACPA pruning, and JSONL output.


In [ ]:
from acpa_gemma.cli import main as pipeline_main

pipeline_main([
    '--config', str(config_dir / 'app.toml'),
    '--secrets', str(config_dir / 'secrets.toml'),
    '--input', str(DATASET_DIR),
    '--output', '/kaggle/working/dry_run.jsonl',
    '--sample-size', '3',
    '--dry-run',
])

print(Path('/kaggle/working/dry_run.jsonl').read_text(encoding='utf-8')[:2000])


## Benchmark ACPA vs baseline algorithms, no API key required

Creates per-record CSV, aggregate CSV, and a Markdown report.


In [ ]:
from acpa_gemma.benchmark import main as benchmark_main

benchmark_main([
    '--input', str(DATASET_DIR),
    '--sample-size', '100',
    '--details-output', '/kaggle/working/benchmark_details.csv',
    '--summary-output', '/kaggle/working/benchmark_summary.csv',
    '--report-output', '/kaggle/working/benchmark_report.md',
])

print(Path('/kaggle/working/benchmark_report.md').read_text(encoding='utf-8'))


## Real Gemma 4 run

Run this after `API key present = True`. Start with a small sample, then remove `--sample-size` for the full run.


In [ ]:
from acpa_gemma.cli import main as pipeline_main

if not api_key:
    raise RuntimeError('Add GEMINI_API_KEY/GEMMA_API_KEY in Kaggle Secrets or paste api_key in the config cell first.')

pipeline_main([
    '--config', str(config_dir / 'app.toml'),
    '--secrets', str(config_dir / 'secrets.toml'),
    '--input', str(DATASET_DIR),
    '--output', '/kaggle/working/results_sample.jsonl',
    '--sample-size', '10',
])

print(Path('/kaggle/working/results_sample.jsonl').read_text(encoding='utf-8')[:3000])


## Full results run

Uncomment this cell when the sample run looks good.


In [ ]:
# pipeline_main([
#     '--config', str(config_dir / 'app.toml'),
#     '--secrets', str(config_dir / 'secrets.toml'),
#     '--input', str(DATASET_DIR),
#     '--output', '/kaggle/working/results.jsonl',
# ])


## Output files

Attach these files to your research report, hackathon demo, or publication appendix as needed:

- `/kaggle/working/results.jsonl`
- `/kaggle/working/results_sample.jsonl`
- `/kaggle/working/benchmark_report.md`
- `/kaggle/working/benchmark_summary.csv`
- `/kaggle/working/benchmark_details.csv`
